In [1]:
import sqlite3
import pandas as pd

# 1. Create a temporary database and 3 distinct tables
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create and populate Customers
cursor.execute('CREATE TABLE Customers (CustomerID INTEGER, CustomerName TEXT, Region TEXT)')
cursor.executemany('INSERT INTO Customers VALUES (?,?,?)', [
    (1, 'Acme Corp', 'North'), (2, 'Global Tech', 'South'), (3, 'Retail Hub', 'East')
])

# Create and populate Products
cursor.execute('CREATE TABLE Products (ProductID INTEGER, ProductName TEXT, Category TEXT, Price REAL)')
cursor.executemany('INSERT INTO Products VALUES (?,?,?,?)', [
    (101, 'Laptop', 'Electronics', 1200.0), (102, 'Desk Chair', 'Furniture', 150.0), (103, 'Monitor', 'Electronics', 300.0)
])

# Create and populate Orders
cursor.execute('CREATE TABLE Orders (OrderID INTEGER, CustomerID INTEGER, ProductID INTEGER, Quantity INTEGER, OrderDate TEXT)')
cursor.executemany('INSERT INTO Orders VALUES (?,?,?,?,?)', [
    (1001, 1, 101, 5, '2026-09-01'), (1002, 1, 102, 10, '2026-09-02'),
    (1003, 2, 103, 20, '2026-09-05'), (1004, 3, 101, 2, '2026-09-10'),
    (1005, 1, 103, 5, '2026-09-15')
])
conn.commit()

# 2. THE MASTER JOIN SCRIPT
# This stitches the 3 tables together and calculates Total Sales
sql_script = """
SELECT
    o.OrderID,
    o.OrderDate,
    c.CustomerName,
    c.Region,
    p.ProductName,
    p.Category,
    o.Quantity,
    p.Price,
    (o.Quantity * p.Price) AS TotalSales
FROM Orders o
INNER JOIN Customers c ON o.CustomerID = c.CustomerID
INNER JOIN Products p ON o.ProductID = p.ProductID
ORDER BY TotalSales DESC;
"""

df_sales = pd.read_sql_query(sql_script, conn)
print("--- CONSOLIDATED SALES VIEW ---")
print(df_sales)

# 3. Export for Power BI
df_sales.to_csv('consolidated_sales.csv', index=False)
print("\nExported to 'consolidated_sales.csv' for Power BI!")

--- CONSOLIDATED SALES VIEW ---
   OrderID   OrderDate CustomerName Region ProductName     Category  Quantity  \
0     1001  2026-09-01    Acme Corp  North      Laptop  Electronics         5   
1     1003  2026-09-05  Global Tech  South     Monitor  Electronics        20   
2     1004  2026-09-10   Retail Hub   East      Laptop  Electronics         2   
3     1002  2026-09-02    Acme Corp  North  Desk Chair    Furniture        10   
4     1005  2026-09-15    Acme Corp  North     Monitor  Electronics         5   

    Price  TotalSales  
0  1200.0      6000.0  
1   300.0      6000.0  
2  1200.0      2400.0  
3   150.0      1500.0  
4   300.0      1500.0  

Exported to 'consolidated_sales.csv' for Power BI!
